<a href="https://colab.research.google.com/github/EaterOfWorlds69/Network-Security-Pre-SOP-Task/blob/main/Network_Security_Pre_SOP_Task.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import io
import pandas as pd
from google.colab import files

uploaded = files.upload()

# Access files by their base name as stored in Colab
df1 = pd.read_csv(io.BytesIO(uploaded["part-00000-363d1ba3-8ab5-4f96-bc25-4d5862db7cb9-c000.csv"]))
df2 = pd.read_csv(io.BytesIO(uploaded["part-00001-363d1ba3-8ab5-4f96-bc25-4d5862db7cb9-c000.csv"]))

print("df1 head:")
display(df1.head())
print("\ndf2 head:")
display(df2.head())

Saving part-00000-363d1ba3-8ab5-4f96-bc25-4d5862db7cb9-c000.csv to part-00000-363d1ba3-8ab5-4f96-bc25-4d5862db7cb9-c000.csv
Saving part-00001-363d1ba3-8ab5-4f96-bc25-4d5862db7cb9-c000.csv to part-00001-363d1ba3-8ab5-4f96-bc25-4d5862db7cb9-c000.csv
df1 head:


,flow_duration,Header_Length,Protocol Type,Duration,Rate,Srate,Drate,fin_flag_number,syn_flag_number,rst_flag_number,...,Std,Tot size,IAT,Number,Magnitue,Radius,Covariance,Variance,Weight,label
0,0.000000,54.00,6.00,64.00,0.329807,0.329807,0.0,1.0,0.0,1.0,...,0.000000,54.00,8.334383e+07,9.5,10.392305,0.000000,0.000000,0.00,141.55,DDoS-RSTFINFlood
1,0.000000,57.04,6.33,64.00,4.290556,4.290556,0.0,0.0,0.0,0.0,...,2.822973,57.04,8.292607e+07,9.5,10.464666,4.010353,160.987842,0.05,141.55,DoS-TCP_Flood
2,0.000000,0.00,1.00,64.00,33.396799,33.396799,0.0,0.0,0.0,0.0,...,0.000000,42.00,8.312799e+07,9.5,9.165151,0.000000,0.000000,0.00,141.55,DDoS-ICMP_Flood
3,0.328175,76175.00,17.00,64.00,4642.133010,4642.133010,0.0,0.0,0.0,0.0,...,0.000000,50.00,8.301570e+07,9.5,10.000000,0.000000,0.000000,0.00,141.55,DoS-UDP_Flood
4,0.117320,101.73,6.11,65.91,6.202211,6.202211,0.0,0.0,1.0,0.0,...,23.113111,57.88,8.297300e+07,9.5,11.346876,32.716243,3016.808286,0.19,141.55,DoS-SYN_Flood



df2 head:


,flow_duration,Header_Length,Protocol Type,Duration,Rate,Srate,Drate,fin_flag_number,syn_flag_number,rst_flag_number,...,Std,Tot size,IAT,Number,Magnitue,Radius,Covariance,Variance,Weight,label
0,0.000000,53.46,5.94,63.36,1.145800,1.145800,0.0,0.0,1.0,0.0,...,0.040218,54.06,8.309402e+07,9.5,10.393232,0.057331,0.054789,0.03,141.55,DDoS-SYN_Flood
1,0.000000,54.00,6.00,64.00,1.027823,1.027823,0.0,0.0,0.0,0.0,...,0.000000,54.00,8.307159e+07,9.5,10.392305,0.000000,0.000000,0.00,141.55,DDoS-TCP_Flood
2,2.204616,93.96,6.00,64.00,0.671213,0.671213,0.0,0.0,1.0,0.0,...,0.000000,54.00,8.336561e+07,9.5,10.392305,0.000000,0.000000,0.00,141.55,DDoS-SynonymousIP_Flood
3,0.053618,12497.00,17.00,64.00,47647.897124,47647.897124,0.0,0.0,0.0,0.0,...,0.000000,50.00,8.310693e+07,9.5,10.000000,0.000000,0.000000,0.00,141.55,DDoS-UDP_Flood
4,0.000000,0.00,1.00,64.00,0.667744,0.667744,0.0,0.0,0.0,0.0,...,0.000000,42.00,8.314933e+07,9.5,9.165151,0.000000,0.000000,0.00,141.55,DDoS-ICMP_Flood


In [ ]:
df_combined = pd.concat([df1, df2], ignore_index=True)
df_combined.to_csv("combined_dataset.csv", index=False)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

switch_features = [
    'Header_Length', 'Protocol Type', 'Duration',
    'TCP', 'UDP', 'ICMP',
    'syn_flag_number', 'ack_flag_number', 'rst_flag_number',
    'fin_flag_number', 'psh_flag_number',
    'Min', 'Max', 'IAT'
]

df_clean = df_combined[switch_features + ['label']].replace([float('inf'), float('-inf')], float('nan')).dropna()

X = df_clean[switch_features]
Y = (df_clean['label'] != 'BenignTraffic').astype(int)

X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size=0.2, random_state=42, stratify=Y
)

clf = DecisionTreeClassifier(max_depth=6, class_weight='balanced', random_state=42)
clf.fit(X_train, Y_train)

print("Training complete, tree depth:", clf.get_depth())

Training complete, tree depth: 6


In [ ]:
from sklearn.ensemble import RandomForestClassifier
import numpy as np

rfc = RandomForestClassifier(n_estimators=20, max_depth=6, class_weight='balanced', random_state=42)
rfc.fit(X_train, Y_train)

depths = [tree.get_depth() for tree in rfc.estimators_]

print(f"Max depth: {max(depths)}")
print(f"Average depth: {np.mean(depths):.2f}")

Max depth: 6
Average depth: 6.00


In [ ]:
from sklearn.metrics import classification_report

print(classification_report(Y_test, clf.predict(X_test)))

              precision    recall  f1-score   support

           0       0.57      1.00      0.72      2160
           1       1.00      0.98      0.99     89339

    accuracy                           0.98     91499
   macro avg       0.78      0.99      0.86     91499
weighted avg       0.99      0.98      0.98     91499

